![Imgur](https://i.imgur.com/A0T6cjl.png)
### [<font size = 2 color="#5890FF">SERIES 4</font>](https://www.kaggle.com/samybaladram/golf-data-analysis-table-of-contents)<font size = 2 color="#7B8698">　‣　C L A S S I F I C A T I O N　🌦️<br></font>[Dummy Classifier](https://www.kaggle.com/samybaladram/04-1-classification-dummy-classifier)・[kNN Classifier](https://www.kaggle.com/samybaladram/04-2-classification-knn-classifier)・[Naive Bayes](https://www.kaggle.com/code/samybaladram/04-3-classification-naive-bayes-classifier)・[Decision Tree](https://www.kaggle.com/samybaladram/04-4-classification-decisiontree-classifier)・**Logistic Regression**<p>
---

# About Linear Model & Logistic Regression
![Imgur](https://i.imgur.com/kLwYHzM.png)
![Imgur](https://i.imgur.com/qvbjF6K.png)

# Main Mechanism
![Imgur](https://i.imgur.com/3f3T0oq.png)

# 📊　Dataset & Libraries Used
![Imgur](https://i.imgur.com/0kF8nSS.png)

In [ ]:
# IMPORTING DATASET #
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
plt.rcParams['figure.dpi'] = 120

dataset_url = "Data/golf_dataset_mini_original_with_testset.csv"
df = pd.read_csv(dataset_url)

df

# Algorithm Steps

# 　Preparation Phase
![Imgur](https://i.imgur.com/MhPcHEj.png)

In [ ]:
df = pd.read_csv(dataset_url)

# Standardize float columns
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
float_cols = df.select_dtypes(include=['float64']).columns
df[float_cols] = scaler.fit_transform(df[float_cols])

df.head()

![Imgur](https://i.imgur.com/1OJG7LT.png)

In [ ]:
from sklearn.model_selection import train_test_split

# Set feature matrix X and target vector y
X, y = df.drop('Play', axis=1), df['Play']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.5, shuffle=False)

# Displaying the train and test set
print(f"X_train, y_train:\n{pd.concat([X_train.round(2), y_train], axis=1)}\n")
print(f"X_test, y_test:\n{pd.concat([X_test.round(2), y_test], axis=1)}")

# 　Training Phase

![Imgur](https://i.imgur.com/1lBtk3d.png)

In [ ]:
weights_0 = np.array([0.1, 0.1, 0.1, 0.1, 0.1, 0.1])
bias_0 = 0

![Imgur](https://i.imgur.com/5E0Qhfb.png)
![Imgur](https://i.imgur.com/8rJUqGh.png)

In [ ]:
# Calculate z-value and probability value
def calculate_z_prob(X_train, weights, bias):
    z_value = np.dot(X_train, weights) + bias
    prob_value = 1 / (1 + np.exp(-z_value))
    df = pd.DataFrame({'z_value': z_value, 'prob_value': prob_value})
    return df

prob_df_0 = calculate_z_prob(X_train, weights_0, bias_0)
print(prob_df_0.round(3))

![Imgur](https://i.imgur.com/KVLdOlN.png)

In [ ]:
def calculate_log_formula(df_z_prob, y_label):
    log_df = pd.DataFrame()
    log_df['prob_value'] = df_z_prob['prob_value']
    log_df['label'] = y_label
    log_df['log formulas'] = [
        f"- log({df_z_prob.loc[idx, 'prob_value']})"
        if y == 1 else f"log(1 - {df_z_prob.loc[idx, 'prob_value']})"
        for idx, y in y_label.items()]
    return log_df

log_df_0 = calculate_log_formula(prob_df_0, y_train)

print(log_df_0.round(3))

In [ ]:
def calculate_loss(prob_df, y_label):
    y_i = np.array(y_label)
    p_i = np.array(prob_df['prob_value'])
    log_loss = -(1/len(y_i))*np.sum(y_i * np.log(p_i) + (1-y_i) * np.log(1-p_i))
    return log_loss

loss = calculate_loss(prob_df_0, y_train)
print(f"Log Loss: {loss.round(3)}")

![Imgur](https://i.imgur.com/3Ham8U1.png)
![Imgur](https://i.imgur.com/ZDhDbLP.png)


In [ ]:
def update_weights(X_train, y_train, weights, bias, learning_rate=69, momentum=0.9, prev_dw=None, prev_db=None):
    m = len(y_train)

    # Compute the predicted probabilities
    z_value = np.dot(X_train, weights) + bias
    pred_probs = 1 / (1 + np.exp(-z_value))

    # Compute the errors
    errors = y_train - pred_probs

    # Calculate gradients
    dw = np.dot(errors, X_train) / m
    db = np.mean(errors)

    # Initialize if it's the first iteration
    if prev_dw is None:
        prev_dw = np.zeros_like(dw)
    if prev_db is None:
        prev_db = 0

    # Momentum term
    dw = momentum * prev_dw + (1 - momentum) * dw
    db = momentum * prev_db + (1 - momentum) * db

    # Adaptive learning rate
    adaptive_lr = learning_rate / (1 + 0.001 * m)

    # Update weights and bias
    new_weights = weights + adaptive_lr * dw
    new_bias = bias + adaptive_lr * db

    return new_weights, new_bias, dw, db

weights = np.array([0.1] * X_train.shape[1])  # Assuming weights_0 is an array of 0.1s
bias = 0
prev_dw, prev_db = None, None
prob_df = calculate_z_prob(X_train, weights, bias)
loss = calculate_loss(prob_df, y_train)

for i in range(101):
    print(f"Weights & Bias {i}: {weights.round(2)}, {round(bias,2)}")
    print(f"Log Loss {i}: {loss.round(3)}\n")
    weights, bias, prev_dw, prev_db = update_weights(X_train, y_train, weights, bias, prev_dw=prev_dw, prev_db=prev_db)
    prob_df = calculate_z_prob(X_train, weights, bias)
    loss = calculate_loss(prob_df, y_train)

In [ ]:
# With sklearn, you can get the final weights (coefficients)
# and final bias (intercepts) easily.
# The result is almost the same as doing it manually above.

from sklearn.linear_model import LogisticRegression

lr_clf = LogisticRegression(penalty=None)
lr_clf.fit(X_train, y_train)

coefficients = lr_clf.coef_
intercept = lr_clf.intercept_

y_train_prob = lr_clf.predict_proba(X_train)[:, 1]
loss = -np.mean(y_train * np.log(y_train_prob) + (1 - y_train) * np.log(1 - y_train_prob))

print(f"Weights & Bias Final: {coefficients[0].round(2)}, {round(intercept[0],2)}")
print("Loss Final:", loss.round(3))

# 　Classification Phase
![Imgur](https://i.imgur.com/TiarAAF.png)

In [ ]:
predicted_probs = lr_clf.predict_proba(X_test)[:, 1]

z_values = np.log(predicted_probs / (1 - predicted_probs))

result_df = pd.DataFrame({
    'ID': X_test.index,
    'Z-Values': z_values.round(3),
    'Probabilities': predicted_probs.round(3)
}).set_index('ID')

print(result_df)

![Imgur](https://i.imgur.com/jcoVrWa.png)

In [ ]:
# Make predictions
y_pred = lr_clf.predict(X_test)
print(y_pred)

# 　Evaluation Phase
![Imgur](https://i.imgur.com/pmJdw08.png)

In [ ]:
result_df = pd.DataFrame({
    'ID': X_test.index,
    'Label': y_test,
    'Probabilities': predicted_probs.round(2),
    'Prediction': y_pred,
}).set_index('ID')

print(result_df)

In [ ]:
# Evaluate the classifier
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")

# Key Parameters

# 　Regularization Type
![Imgur](https://i.imgur.com/kHd5xmN.png)
![Imgur](https://i.imgur.com/zbQJc3m.png)
![Imgur](https://i.imgur.com/ylUF71p.png)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

regs = [None, 'l1', 'l2']
coeff_dict = {}

for reg in regs:
    lr_clf = LogisticRegression(penalty=reg, solver='saga')
    lr_clf.fit(X_train, y_train)
    coefficients = lr_clf.coef_
    intercept = lr_clf.intercept_
    predicted_probs = lr_clf.predict_proba(X_train)[:, 1]
    loss = -np.mean(y_train * np.log(predicted_probs) + (1 - y_train) * np.log(1 - predicted_probs))
    predictions = lr_clf.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    coeff_dict[reg] = {
        'Coefficients': coefficients,
        'Intercept': intercept,
        'Loss': loss,
        'Accuracy': accuracy
    }

for reg, vals in coeff_dict.items():
    print(f"{reg}: Coeff: {vals['Coefficients'][0].round(2)}, Intercept: {vals['Intercept'].round(2)}, Loss: {vals['Loss'].round(3)}, Accuracy: {vals['Accuracy'].round(3)}")


# 　Regularization Strength
![Imgur](https://i.imgur.com/L64yX8f.png)
![Imgur](https://i.imgur.com/UKOa8Tz.png)

In [ ]:
# List of regularization strengths to try
strengths = [0.001, 0.01, 0.1, 1, 10, 100]

coeff_dict = {}

for strength in strengths:
    lr_clf = LogisticRegression(penalty='l1', C=strength, solver='saga')
    lr_clf.fit(X_train, y_train)

    coefficients = lr_clf.coef_
    intercept = lr_clf.intercept_

    predicted_probs = lr_clf.predict_proba(X_train)[:, 1]
    loss = -np.mean(y_train * np.log(predicted_probs) + (1 - y_train) * np.log(1 - predicted_probs))
    predictions = lr_clf.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)

    coeff_dict[f'L1_{strength}'] = {
        'Coefficients': coefficients[0].round(2),
        'Intercept': round(intercept[0],2),
        'Loss': round(loss,3),
        'Accuracy': round(accuracy*100,2)
    }

print(pd.DataFrame(coeff_dict).T)

![Imgur](https://i.imgur.com/xHu4BWJ.png)

In [ ]:
# List of regularization strengths to try
strengths = [0.001, 0.01, 0.1, 1, 10, 100]

coeff_dict = {}

for strength in strengths:
    lr_clf = LogisticRegression(penalty='l2', C=strength, solver='saga')
    lr_clf.fit(X_train, y_train)

    coefficients = lr_clf.coef_
    intercept = lr_clf.intercept_

    predicted_probs = lr_clf.predict_proba(X_train)[:, 1]
    loss = -np.mean(y_train * np.log(predicted_probs) + (1 - y_train) * np.log(1 - predicted_probs))
    predictions = lr_clf.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    coeff_dict[f'L2_{strength}'] = {
        'Coefficients': coefficients[0].round(2),
        'Intercept': round(intercept[0],2),
        'Loss': round(loss,3),
        'Accuracy': round(accuracy*100,2)
    }

print(pd.DataFrame(coeff_dict).T)

# Pros and Cons
![Imgur](https://i.imgur.com/gFFEFY1.png)

# 🌟 Logistic Regression Simplified

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(dataset_url)

# Standardize numerical columns
scaler = StandardScaler()
float_cols = df.select_dtypes(include=['float64']).columns
df[float_cols] = scaler.fit_transform(df[float_cols])

# Split train and test data
X, y = df.drop('Play', axis=1), df['Play']
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.5, shuffle=False)

# Fit the model
lr_clf = LogisticRegression(penalty='l2', C=1, solver='saga') # Penalty: None, 'l1', 'l2'
lr_clf.fit(X_train, y_train)

# Predict the test data
y_pred = lr_clf.predict(X_test)

# Evaluate the model
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")

![Imgur](https://i.imgur.com/M0mwv0l.png)
### [<font size = 2 color="#5890FF">SERIES 4</font>](https://www.kaggle.com/samybaladram/golf-data-analysis-table-of-contents)<font size = 2 color="#7B8698">　‣　C L A S S I F I C A T I O N　🌦️<br></font>[Dummy Classifier](https://www.kaggle.com/samybaladram/04-1-classification-dummy-classifier)・[kNN Classifier](https://www.kaggle.com/samybaladram/04-2-classification-knn-classifier)・[Naive Bayes](https://www.kaggle.com/code/samybaladram/04-3-classification-naive-bayes-classifier)・[Decision Tree](https://www.kaggle.com/samybaladram/04-4-classification-decisiontree-classifier)・**Logistic Regression**<p>
    
--- 

*Explore [diverse notebooks full of captivating visuals](https://www.kaggle.com/samybaladram/golf-data-analysis-table-of-contents), each tackling unique facets of the field—all based on some mini, yet enriching, golf play datasets. Your hole-in-one guide to mastering Data Science basics.*


### For a complete list of topics and series, see [⛳️ 📋 Data Science with Golf Dataset・Table of Contents](https://www.kaggle.com/samybaladram/golf-data-analysis-table-of-contents).
---

**Your Feedback Matters! 🍃** If you find this notebook series useful, an ⬆️ *upvote* would be appreciated to help others in the community discover it too!